# Simulating a quantum circuit using the density matrix formalism


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# helper functions for manipulating and visualizing density matrices
from graphiq.visualizers.density_matrix import density_matrix_bars
from graphiq.backends.density_matrix.functions import (
    state_ketz0,
    state_ketz1,
    tensor,
    ket2dm,
    partial_trace,
    fidelity,
)

First, we create the quantum circuit by adding gates, specifying the registers (quantum and classical) that they act on.

In [ ]:
from graphiq.circuit.circuit_dag import CircuitDAG

# import the relevant operations to add to the circuit
from graphiq.circuit.ops import Hadamard, CNOT, MeasurementCNOTandReset

# build up the quantum circuit for a 4-qubit GHZ state
circuit = CircuitDAG(n_emitter=1, n_photon=4, n_classical=1)
circuit.add(Hadamard(register=0, reg_type="e"))
circuit.add(CNOT(control=0, control_type="e", target=0, target_type="p"))
circuit.add(CNOT(control=0, control_type="e", target=1, target_type="p"))
circuit.add(CNOT(control=0, control_type="e", target=2, target_type="p"))

circuit.add(CNOT(control=0, control_type="e", target=3, target_type="p"))
circuit.add(Hadamard(register=3, reg_type="p"))
circuit.add(Hadamard(register=0, reg_type="e"))

circuit.add(
    MeasurementCNOTandReset(
        control=0, control_type="e", target=3, target_type="p", c_register=0
    )
)
circuit.add(Hadamard(register=3, reg_type="p"))

In [ ]:
# plot the circuit as a directed, acyclic graph
circuit.draw_dag()

Now, we select a backend compiler to simulate the evolution of the quantum circuit. We currently can do this for density matrices, simulating the circuit effect by applying unitary gates.

In [ ]:
from graphiq.backends.density_matrix.compiler import DensityMatrixCompiler

compiler = DensityMatrixCompiler()
state = compiler.compile(
    circuit
)  # this returns a QuantumState object, and you can get underlying complex array as state.rep_data.data

In [ ]:
# trace out the ancilla qubit
state_a = partial_trace(state.rep_data.data, keep=(0, 1, 2, 3), dims=5 * [2])

In [ ]:
fig, ax = density_matrix_bars(state_a)
fig.suptitle("Simulated circuit density matrix")
plt.show()

In [ ]:
# compare to the ideal density matrix
ideal_state = ket2dm(
    (tensor(4 * [state_ketz0()]) + tensor(4 * [state_ketz1()])) / np.sqrt(2)
)
f = fidelity(state_a, ideal_state)
print(f"Fidelity with ideal state: {f}")

In [ ]:
# save the resulting quantum state
from graphiq.io import IO

io = IO.new_directory(folder="example-folder", include_date=True, include_id=True)
io.save_np_array(state_a, "partial_density_matrix.np")

Future work will build metrics (e.g., fidelity, circuit depth, success probability) and solvers (e.g., random search, decompositions, etc.), as well as other backend compilers.

In [ ]:
from graphiq import metrics
from graphiq.state import QuantumState
from graphiq.solvers.evolutionary_solver import EvolutionarySolver, EvolutionarySolverSetting

In [ ]:
target = QuantumState(
    ket2dm((tensor(4 * [state_ketz0()]) + tensor(4 * [state_ketz1()])) / np.sqrt(2)),
    rep_type="dm",
)

# we pass one metric to use as the cost function. we can also pass more to be evaluated but not used as the cost
metric = metrics.Infidelity(target)

# define the solver (all methods are encapsulated in the class definition)
solver = EvolutionarySolver(
    target=target,
    metric=metric,
    compiler=compiler,
    circuit=None,
    n_emitter=1,
    n_photon=4,
    solver_setting=EvolutionarySolverSetting(tournament_k=0),
)

# call .solve to implement the solver algorithm
solver.solve()

# we now have access to the metrics (don't need to pass anything back, as we are logging it in the Metric instance)
print(metric.log)

plt.figure()
plt.plot(metric.log)
plt.show()